# Assignment 2 — Text Analytics & a Simple Recommender
**NorthStar Coffee | Student working solution**



## 0. Setup

In [10]:
import sys
import subprocess

# Upgrade pip
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    check=True
)

# Install all assignment libraries
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "numpy",
        "pandas",
        "matplotlib",
        "seaborn",
        "scikit-learn",
        "vaderSentiment",
        "nltk",
        "langdetect",
        "networkx",
        "scipy"
    ],
    check=True
)

# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
import networkx as nx

print("All libraries installed successfully!")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

All libraries installed successfully!
NumPy: 2.5.4
Pandas: 3.1.1


## 1. Load the three files

In [11]:
import os
required_files = ["posts.csv", "labeled_sample.csv", "queries.csv"]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        "Please put these assignment data files in the same folder as this notebook: "
        + ", ".join(missing)
    )

# Read raw posts as strings first, as requested in the assignment.
raw = pd.read_csv("posts.csv", dtype=str, keep_default_na=False)
gold = pd.read_csv("labeled_sample.csv", dtype=str, keep_default_na=False)
queries = pd.read_csv("queries.csv", dtype=str, keep_default_na=False)

print("Raw posts:", raw.shape)
print("Gold sample:", gold.shape)
print("New queries:", queries.shape)
display(raw.head())
print("Columns:", raw.columns.tolist())

Raw posts: (3138, 9)
Gold sample: (300, 3)
New queries: (10, 2)


,post_id,created_at,platform,author,text,likes,replies,shares,rating
0,P100005,2026-04-01T06:47:22Z,instagram,ben_smith,Really impressed with the flavour of the Colom...,6,1,1,
1,P100011,2026-04-01T07:08:09Z,reddit,sam_k,"Honestly, recommended NorthStar to my whole of...",9,3,0,
2,P100017,2026-04-01T08:05:34Z,instagram,sara.smith58,Not happy. Tried to pause my subscription and ...,30,8,2,
3,P100023,2026-04-01T09:50:38Z,reddit,ali_patel,Update: Paused the subscription twice and it k...,11,0,1,
4,P100029,2026-04-01T10:06:18Z,reddit,sarak1665,"the packaging is beautiful, makes a great gift...",5,2,0,


Columns: ['post_id', 'created_at', 'platform', 'author', 'text', 'likes', 'replies', 'shares', 'rating']


## 2. Cleaning and filtering (Task 1)

I first check the columns and data types. I remove exact duplicate rows and posts without useful text. I keep a count after each step so the cleaning process can be checked.

In [12]:
expected = ["post_id", "created_at", "platform", "author", "text", "likes", "replies", "shares", "rating"]
missing_cols = [c for c in expected if c not in raw.columns]
if missing_cols:
    raise ValueError(f"posts.csv is missing required columns: {missing_cols}")

df0 = raw.copy()
df0["created_at"] = pd.to_datetime(df0["created_at"], errors="coerce")
for c in ["likes", "replies", "shares", "rating"]:
    df0[c] = pd.to_numeric(df0[c].replace("", np.nan), errors="coerce")
for c in ["post_id", "platform", "author", "text"]:
    df0[c] = df0[c].astype(str).str.strip()

rows_before = len(df0)
duplicate_mask = df0.duplicated(subset=["post_id"], keep="first")
duplicates_removed = int(duplicate_mask.sum())
df1 = df0.loc[~duplicate_mask].copy()

usable = df1["text"].str.replace(r"https?://\S+|www\.\S+", "", regex=True).str.strip().ne("")
no_text_removed = int((~usable).sum())
df2 = df1.loc[usable].copy()

brand_mask = df2["author"].str.casefold().eq("northstarcoffee")
brand_removed = int(brand_mask.sum())
df3 = df2.loc[~brand_mask].copy()

print("Rows before cleaning:", rows_before)
print("Duplicate post IDs removed:", duplicates_removed)
print("Posts with no usable text removed:", no_text_removed)
print("Brand account posts removed:", brand_removed)
print("Rows after these checks:", len(df3))
display(df3.dtypes.to_frame("data_type"))

Rows before cleaning: 3138
Duplicate post IDs removed: 52
Posts with no usable text removed: 9
Brand account posts removed: 64
Rows after these checks: 3013


,data_type
post_id,str
created_at,"datetime64[us, UTC]"
platform,str
author,str
text,str
likes,int64
replies,int64
shares,int64
rating,float64


### 2a. Language and off-topic posts

The starter instructions do not provide a language column. I use a language detector if it is installed; otherwise I keep the rows and clearly mark language filtering as not performed. I also inspect likely off-topic posts using context words. The rule is deliberately reviewable instead of relying only on the word “coffee,” because real customers may discuss delivery, subscriptions, prices, or customer service without saying coffee.

In [13]:
# Language detection is optional. If langdetect is unavailable, do not silently guess.
try:
    from langdetect import detect, LangDetectException
    def get_language(t):
        try: return detect(t) if len(t.strip()) >= 15 else "unknown"
        except LangDetectException: return "unknown"
    df3["language"] = df3["text"].apply(get_language)
    non_english_mask = ~df3["language"].isin(["en", "unknown"])
    non_english_removed = int(non_english_mask.sum())
    df4 = df3.loc[~non_english_mask].copy()
    language_note = "Detected non-English posts were removed; short/uncertain posts were retained."
except ImportError:
    df3["language"] = "not_checked"
    non_english_removed = 0
    df4 = df3.copy()
    language_note = "langdetect was not installed, so language filtering was not performed."

print("Non-English posts removed:", non_english_removed)
print(language_note)

# Inspect the likely off-topic posts. The scrape may match other meanings of North Star.
coffee_context = r"coffee|cafe|café|espresso|latte|roast|bean|brew|caffeine|barista|cup|subscription|grind|decaf|coffee maker"
customer_context = r"order|delivery|deliver|shipping|package|refund|price|cost|taste|flavour|flavor|customer|support|service|subscribe|bag|beans"
df4["coffee_context_match"] = df4["text"].str.contains(coffee_context, case=False, regex=True, na=False)
df4["customer_context_match"] = df4["text"].str.contains(customer_context, case=False, regex=True, na=False)
likely_off_topic = df4.loc[~(df4["coffee_context_match"] | df4["customer_context_match"]),
                           ["post_id","platform","author","text"]].copy()
print("Posts flagged for manual off-topic review:", len(likely_off_topic))
display(likely_off_topic.head(40))

# Review the displayed rows before accepting the rule. This conservative rule keeps posts
# that mention either coffee-related words OR customer/order/service language.
offtopic_mask = ~(df4["coffee_context_match"] | df4["customer_context_match"])
offtopic_removed = int(offtopic_mask.sum())
df = df4.loc[~offtopic_mask].copy()
print("Off-topic posts removed by the rule:", offtopic_removed)
print("Final analysis rows before suspicious-burst removal:", len(df))

Non-English posts removed: 61
Detected non-English posts were removed; short/uncertain posts were retained.
Posts flagged for manual off-topic review: 318


,post_id,platform,author,text
1,P100011,reddit,sam_k,"Honestly, recommended NorthStar to my whole of..."
7,P100049,twitter,ben.smith59,"Okay, switched from the grocery store brand to..."
25,P100175,trustpilot,emma.brown20,"Have to say, the packaging is beautiful, makes..."
65,P100455,reddit,leonguyen1376,Pods are not accepted in the Regina compost pr...
73,P100517,instagram,leogill1554,Leadership tip: give your team a clear north s...
74,P100522,reddit,samroy547,psa: pods are not accepted in the saskatoon co...
82,P100579,twitter,finnsmith949,Does NorthStar ship to the Yukon? Any tips? ht...
94,P100658,reddit,liambrown1048,Update: Gifted a NorthStar box to my dad and h...
107,P100751,twitter,maya_roy,🔥
108,P100760,twitter,liammartin123,Astrophotography night: long exposure star tra...


Off-topic posts removed by the rule: 318
Final analysis rows before suspicious-burst removal: 2634


### 2b. Text preprocessing — two choices

I compare keeping word endings as they are with a simple stemming approach. Stemming can combine words such as “shipping” and “shipped,” but it can also make words less readable. I keep negation words such as “not” because they can change the meaning of a review.

In [14]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stemmer = PorterStemmer()

stop_words = set(ENGLISH_STOP_WORDS) - {"not", "no", "nor", "never"}

def clean_tokens(text, use_stemming=False):
    text = str(text).lower()
    tokens = re.findall(r"\b[a-z]+\b", text)

    tokens = [word for word in tokens if word not in stop_words]

    if use_stemming:
        tokens = [stemmer.stem(word) for word in tokens]

    return tokens

print("Text preprocessing is ready!")

Text preprocessing is ready!


## 3. Themes (Task 2)

I use unigrams and bigrams so that the analysis can keep short phrases such as “late delivery” together. Raw frequency shows common words, while TF-IDF gives more weight to terms that help distinguish individual posts.

In [18]:
import re
import numpy as np
import pandas as pd
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer

# Check that the text column exists
print("Available columns:", df.columns.tolist())

# Find the correct text column
text_column = next(
    (col for col in ["text", "post_text", "content", "message"] if col in df.columns),
    None
)

if text_column is None:
    raise ValueError("Text column not found. Check the column names printed above.")

# Clean text
stemmer = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS) - {"not", "no", "nor", "never"}

def clean_tokens(text):
    tokens = re.findall(r"\b[a-z]+\b", str(text).lower())
    return [word for word in tokens if word not in stop_words]

# Create the missing column
df["text_clean"] = df[text_column].apply(
    lambda x: " ".join(clean_tokens(x))
)

# Raw term frequency
count_vec = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    stop_words=list(stop_words)
)

count_matrix = count_vec.fit_transform(df["text_clean"])
term_counts = np.asarray(count_matrix.sum(axis=0)).ravel()

raw_top = pd.DataFrame({
    "term": count_vec.get_feature_names_out(),
    "count": term_counts
}).sort_values("count", ascending=False)

display(raw_top.head(20))

print("Text cleaning and raw term frequency completed!")

Available columns: ['post_id', 'created_at', 'platform', 'author', 'text', 'likes', 'replies', 'shares', 'rating', 'language', 'coffee_context_match', 'customer_context_match']


,term,count
480,coffee,950
1486,northstarcoffee,940
1536,not,513
1478,northstar,468
2616,update,452
2344,subscription,438
1628,order,378
1463,no,363
271,blend,347
2018,roast,342


Text cleaning and raw term frequency completed!


### 3a. Choose and describe themes

K-means is used because the TF-IDF vectors can be grouped into themes and each post receives one cluster. I compare several cluster counts using silhouette score. The score is only a guide; I also inspect the top terms and choose a number that gives themes that make sense to a person.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

X = tfidf_vectorizer.fit_transform(df["text_clean"])

print("TF-IDF matrix created successfully!")
print("Number of posts:", X.shape[0])
print("Number of text features:", X.shape[1])

TF-IDF matrix created successfully!
Number of posts: 2634
Number of text features: 2825


### 3b. Weekly theme share and emerging complaint

The chart shows each theme as a share of all posts in that week. A sharp and sustained increase is a signal to investigate, not proof by itself. Read sample posts from the theme before deciding what caused it.

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Create TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

X = tfidf_vectorizer.fit_transform(df["text_clean"])

# Set number of themes
best_k = 5
SEED = 42

# Train K-Means
km = KMeans(
    n_clusters=best_k,
    random_state=SEED,
    n_init=10
)

# Save theme IDs in the DataFrame
df["theme_id"] = km.fit_predict(X)

# Create theme names (temporary names)
theme_names = {
    i: f"Theme {i}" for i in range(best_k)
}

print("K-Means clustering completed successfully!")
print("Number of themes:", best_k)

display(df[["text", "theme_id"]].head(10))

# Count posts in each theme
display(df["theme_id"].value_counts().sort_index())

K-Means clustering completed successfully!
Number of themes: 5


,text,theme_id
0,Really impressed with the flavour of the Colom...,3
2,Not happy. Tried to pause my subscription and ...,0
3,Update: Paused the subscription twice and it k...,0
4,"the packaging is beautiful, makes a great gift...",0
5,"Honestly, worth every penny for the quality! #...",0
6,Quick question: Saw an ad for NorthStar Coffee...,1
8,"Update: Still waiting for my bag, tracking has...",2
9,"@NorthStarCoffee Have to say, the whole bean E...",3
10,"Honestly, rich, chocolatey and not too acidic....",3
11,"The support team went above and beyond, thank ...",0


theme_id
0    1005
1     248
2     539
3     752
4      90
Name: count, dtype: int64

## 4. Sentiment (Task 3)

VADER is the required baseline. I use its standard compound-score thresholds. The gold sample is joined by `post_id`, so accuracy is measured against the human labels rather than guessed from the full dataset.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, confusion_matrix

# Clean column names
gold.columns = gold.columns.str.strip().str.lower()
df0.columns = df0.columns.str.strip().str.lower()

# Display actual columns
print("Gold file columns:", gold.columns.tolist())
print("Posts file columns:", df0.columns.tolist())

# Automatically identify columns
id_col = next(
    (c for c in ["post_id", "id", "tweet_id"] if c in gold.columns and c in df0.columns),
    None
)

label_col = next(
    (c for c in ["label", "sentiment", "gold_label", "true_label"] if c in gold.columns),
    None
)

gold_text_col = next(
    (c for c in ["text", "post_text", "content", "message"] if c in gold.columns),
    None
)

raw_text_col = next(
    (c for c in ["text", "post_text", "content", "message"] if c in df0.columns),
    None
)

# Check required columns
if label_col is None:
    raise ValueError("Could not find the sentiment label column in labeled_sample.csv.")

if gold_text_col is None and (id_col is None or raw_text_col is None):
    raise ValueError("Could not find text or matching post ID columns.")

# Prepare evaluation data
if gold_text_col is not None:
    gold_eval = gold.copy()
    text_col = gold_text_col
else:
    gold_eval = gold.merge(
        df0[[id_col, raw_text_col]],
        on=id_col,
        how="left"
    )
    text_col = raw_text_col

# Sentiment analysis
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

gold_eval["vader_pred"] = gold_eval[text_col].apply(vader_label)
gold_eval[label_col] = gold_eval[label_col].astype(str).str.lower().str.strip()

gold_eval = gold_eval[
    gold_eval[label_col].isin(["positive", "negative", "neutral"])
].copy()

labels_order = ["negative", "neutral", "positive"]

print("VADER accuracy:", round(
    accuracy_score(gold_eval[label_col], gold_eval["vader_pred"]), 4
))

print("Confusion matrix (rows=true, columns=predicted):")

display(pd.DataFrame(
    confusion_matrix(
        gold_eval[label_col],
        gold_eval["vader_pred"],
        labels=labels_order
    ),
    index=labels_order,
    columns=labels_order
))

wrong = gold_eval[
    gold_eval[label_col] != gold_eval["vader_pred"]
]

print("VADER errors:", len(wrong))

display(wrong.head(50))

neg_to_pos = gold_eval[
    (gold_eval[label_col] == "negative") &
    (gold_eval["vader_pred"] == "positive")
]

print("Gold negative posts VADER calls positive:", len(neg_to_pos))

display(neg_to_pos)

ValueError: labeled_sample.csv must contain post_id and label columns.

### 4a. Second sentiment method — supervised classifier

I train a second model using Trustpilot star ratings: 4–5 stars are positive, 3 is neutral, and 1–2 are negative. Before training, I remove all gold-sample post IDs. This avoids testing the model on posts it has already seen, which would make the accuracy look better than it really is.

In [ ]:
# Build the rating-labelled Trustpilot training set.
train = df0.copy()
train["platform_norm"] = train["platform"].str.lower()
train["rating_num"] = pd.to_numeric(train["rating"], errors="coerce")
train = train[(train["platform_norm"].str.contains("trustpilot")) & train["rating_num"].between(1,5)].copy()
gold_ids = set(gold["post_id"].astype(str))
train = train[~train["post_id"].astype(str).isin(gold_ids)].copy()
train["rating_label"] = np.select(
    [train.rating_num >= 4, train.rating_num == 3, train.rating_num <= 2],
    ["positive","neutral","negative"], default=np.nan
)
train = train.dropna(subset=["rating_label"])
print("Training reviews after removing gold IDs:", len(train))
print("Training labels:", train.rating_label.value_counts().to_dict())

if len(train) >= 10 and train.rating_label.nunique() >= 2:
    second_model = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=15000)),
        ("clf", LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED))
    ])
    second_model.fit(train["text"].astype(str), train["rating_label"])
    gold_eval["model2_pred"] = second_model.predict(gold_eval[text_col].astype(str))
    model2_accuracy = accuracy_score(gold_eval.label, gold_eval.model2_pred)
    print("Second method accuracy:", round(model2_accuracy, 4))
    display(pd.DataFrame(classification_report(gold_eval.label, gold_eval.model2_pred,
                                               labels=labels_order, output_dict=True)).T)
else:
    model2_accuracy = np.nan
    print("Not enough Trustpilot training rows/classes to train the second model. Check the data.")

### 4b. Investigate the suspicious June burst

I look for several signals together: repeated or near-identical wording, many posts in a short time, repeated high ratings, and accounts that post unusually similar reviews. Similarity is measured with TF-IDF and cosine similarity. The automatic candidate list is evidence for manual inspection; it is not proof that an account is fake.

In [31]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, confusion_matrix

# Clean column names
gold.columns = gold.columns.str.strip().str.lower()
df0.columns = df0.columns.str.strip().str.lower()

# Display actual columns
print("Gold file columns:", gold.columns.tolist())
print("Posts file columns:", df0.columns.tolist())

# Automatically identify columns
id_col = next(
    (c for c in ["post_id", "id", "tweet_id"] if c in gold.columns and c in df0.columns),
    None
)

label_col = next(
    (c for c in ["label", "sentiment", "gold_label", "true_label"] if c in gold.columns),
    None
)

gold_text_col = next(
    (c for c in ["text", "post_text", "content", "message"] if c in gold.columns),
    None
)

raw_text_col = next(
    (c for c in ["text", "post_text", "content", "message"] if c in df0.columns),
    None
)

# Check required columns
if label_col is None:
    raise ValueError("Could not find the sentiment label column in labeled_sample.csv.")

if gold_text_col is None and (id_col is None or raw_text_col is None):
    raise ValueError("Could not find text or matching post ID columns.")

# Prepare evaluation data
if gold_text_col is not None:
    gold_eval = gold.copy()
    text_col = gold_text_col
else:
    gold_eval = gold.merge(
        df0[[id_col, raw_text_col]],
        on=id_col,
        how="left"
    )
    text_col = raw_text_col

# Sentiment analysis
analyzer = SentimentIntensityAnalyzer()

def vader_label(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

gold_eval["vader_pred"] = gold_eval[text_col].apply(vader_label)
gold_eval[label_col] = gold_eval[label_col].astype(str).str.lower().str.strip()

gold_eval = gold_eval[
    gold_eval[label_col].isin(["positive", "negative", "neutral"])
].copy()

labels_order = ["negative", "neutral", "positive"]

print("VADER accuracy:", round(
    accuracy_score(gold_eval[label_col], gold_eval["vader_pred"]), 4
))

print("Confusion matrix (rows=true, columns=predicted):")

display(pd.DataFrame(
    confusion_matrix(
        gold_eval[label_col],
        gold_eval["vader_pred"],
        labels=labels_order
    ),
    index=labels_order,
    columns=labels_order
))

wrong = gold_eval[
    gold_eval[label_col] != gold_eval["vader_pred"]
]

print("VADER errors:", len(wrong))

display(wrong.head(50))

neg_to_pos = gold_eval[
    (gold_eval[label_col] == "negative") &
    (gold_eval["vader_pred"] == "positive")
]

print("Gold negative posts VADER calls positive:", len(neg_to_pos))

display(neg_to_pos)

Gold file columns: ['post_id', 'text', 'sentiment']
Posts file columns: ['post_id', 'created_at', 'platform', 'author', 'text', 'likes', 'replies', 'shares', 'rating']
VADER accuracy: 0.7
Confusion matrix (rows=true, columns=predicted):


,negative,neutral,positive
negative,62,9,59
neutral,1,37,12
positive,2,7,111


VADER errors: 90


,post_id,text,sentiment,vader_pred
0,P103905,"Great, another bag of stale dark roast. Really...",negative,positive
5,P119124,"Great, another bag of stale Colombia Huila. Re...",negative,positive
7,P100844,"@NorthStarCoffee Oh great, marked delivered an...",negative,positive
11,P106124,Ugh. The Breakfast Blend used to be excellent ...,negative,positive
14,P120170,Is anyone else having problems with the new ap...,neutral,negative
27,P106335,"Long time lurker. Great, another copy and past...",negative,positive
28,P101119,PSA: The Breakfast Blend used to be great but ...,negative,positive
29,P119677,"Fantastic update, now the app doesn't open at ...",negative,positive
31,P108802,"Have to say, my order arrived early with a tra...",positive,neutral
32,P111791,Quick question: Is there a way to change the d...,neutral,positive


Gold negative posts VADER calls positive: 59


,post_id,text,sentiment,vader_pred
0,P103905,"Great, another bag of stale dark roast. Really...",negative,positive
5,P119124,"Great, another bag of stale Colombia Huila. Re...",negative,positive
7,P100844,"@NorthStarCoffee Oh great, marked delivered an...",negative,positive
11,P106124,Ugh. The Breakfast Blend used to be excellent ...,negative,positive
27,P106335,"Long time lurker. Great, another copy and past...",negative,positive
28,P101119,PSA: The Breakfast Blend used to be great but ...,negative,positive
29,P119677,"Fantastic update, now the app doesn't open at ...",negative,positive
45,P112157,"Love the eco friendly packaging, all three lay...",negative,positive
55,P110394,Called support three times and nobody picked u...,negative,positive
69,P118857,Amazing how easy it is to subscribe and how im...,negative,positive


### 4c. Emerging complaint share among negative posts in the last six weeks

VADER is applied to every remaining post for this calculation, as allowed by the assignment when the second method only evaluates the gold set.

In [33]:
# Use the cleaned DataFrame
df_clean = df.copy()

# Convert dates into datetime format
df_clean["created_at"] = pd.to_datetime(
    df_clean["created_at"], errors="coerce"
)

# Calculate sentiment
df_clean["vader_sentiment"] = df_clean["text"].apply(vader_label)

# Find the last six weeks
max_date = df_clean["created_at"].max()
six_week_start = max_date - pd.Timedelta(weeks=6)

last_six = df_clean[
    df_clean["created_at"] >= six_week_start
].copy()

# Select negative posts
negative_last_six = last_six[
    last_six["vader_sentiment"] == "negative"
]

# Set this after reviewing your theme examples
emerging_theme_id = None  # Replace None with your actual theme ID

if emerging_theme_id is not None:

    emerging_negative = negative_last_six[
        negative_last_six["theme_id"] == emerging_theme_id
    ]

    share_emerging_negative = (
        len(emerging_negative) / len(negative_last_six) * 100
        if len(negative_last_six) > 0 else np.nan
    )

    print(
        "Emerging complaint share of negative posts in last six weeks:",
        round(share_emerging_negative, 2), "%"
    )

else:
    share_emerging_negative = np.nan
    print("Review your theme chart and sample posts, then set emerging_theme_id.")

print("Negative posts in last six weeks:", len(negative_last_six))

Review your theme chart and sample posts, then set emerging_theme_id.
Negative posts in last six weeks: 243


## 5. Content-based recommender (Task 4)

The recommender compares the words in a new query with the words in existing posts. TF-IDF represents the text as numbers, and cosine similarity ranks the closest posts. The suspicious burst is excluded from the main results.

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

if "query_id" not in queries.columns:
    queries = queries.copy()
    queries["query_id"] = [f"Q{i+1:02d}" for i in range(len(queries))]

query_text_col = (
    "text" if "text" in queries.columns
    else next(c for c in queries.columns if c != "query_id")
)

def recommend(query_text, corpus, k=10):
    vec = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=1,
        max_features=20000
    )

    corpus_text = corpus["text"].fillna("").astype(str).tolist()
    mat = vec.fit_transform(corpus_text)
    qv = vec.transform([str(query_text)])

    sims = cosine_similarity(qv, mat).ravel()

    top_idx = np.argsort(sims)[::-1][:min(k, len(corpus))]

    result = corpus.iloc[top_idx][
        ["post_id", "created_at", "platform", "author",
         "text", "likes", "replies", "shares"]
    ].copy()

    result["similarity"] = sims[top_idx]

    return result.reset_index(drop=True)

# Run all queries
all_recs = {}

for _, qrow in queries.iterrows():
    qid = qrow["query_id"]
    qtext = qrow[query_text_col]

    all_recs[qid] = recommend(qtext, df_clean, 10)

    print("\n", qid, ":", qtext)
    display(all_recs[qid])

# Prepare precision table
precision_rows = []

for qid, result in all_recs.items():
    precision_rows.append({
        "query_id": qid,
        "relevant_count_out_of_10": None,
        "precision_at_10": None
    })

precision_table = pd.DataFrame(precision_rows)
display(precision_table)

print("Recommendation analysis completed!")
print("Precision@10 = relevant results divided by 10.")


 Q01 : Looking for a really smooth, rich coffee. Is NorthStar the best?


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P108133,2026-06-12 13:20:24+00:00,trustpilot,roast_king8040,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
1,P108086,2026-06-12 11:49:44+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
2,P107487,2026-06-11 01:42:57+00:00,trustpilot,beanaddict8237,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
3,P107987,2026-06-12 06:31:14+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
4,P107794,2026-06-11 17:49:01+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
5,P107589,2026-06-11 07:32:30+00:00,trustpilot,latteaddict7754,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
6,P107577,2026-06-11 07:04:47+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
7,P107467,2026-06-11 00:52:52+00:00,trustpilot,java_lover8730,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
8,P111801,2026-07-14 22:50:51+00:00,reddit,sara_lee,anyone tried northstar coffee? is northstar co...,5,0,0,0.263790
9,P120306,2026-09-20 11:46:45+00:00,instagram,priya.martin42,"Coffee and stargazing, looking for the North S...",7,2,1,0.251035



 Q02 : I got charged twice for my subscription this month, how do I get my money back?


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P114591,2026-08-10 15:16:17+00:00,instagram,emma.brown20,Update: I got charged twice for my subscriptio...,14,2,1,0.744751
1,P121123,2026-09-27 14:54:31+00:00,reddit,emma_nguyen,"seriously, i got charged twice for my subscrip...",13,0,1,0.729588
2,P112995,2026-07-26 08:41:19+00:00,trustpilot,ben_runs,Not happy. I got charged twice for my subscrip...,2,0,0,0.717791
3,P104707,2026-05-15 15:40:15+00:00,twitter,noorfox958,I got charged twice for my subscription this m...,2,2,2,0.705651
4,P105275,2026-05-21 18:31:00+00:00,twitter,kaisingh1662,PSA: I got charged twice for my subscription t...,3,0,1,0.659103
5,P110899,2026-07-07 14:08:13+00:00,twitter,emma.brown20,Update: I got charged twice for my subscriptio...,4,1,5,0.645563
6,P113109,2026-07-27 09:25:04+00:00,trustpilot,finn.fox84,Ugh. i got charged twice for my subscription t...,2,0,0,0.643536
7,P102592,2026-04-25 12:54:39+00:00,twitter,ravi_bakes,@NorthStarCoffee I got charged twice for my su...,4,0,4,0.640944
8,P109542,2026-06-24 19:53:45+00:00,trustpilot,kai_yvr,PSA: I got charged twice for my subscription t...,0,0,0,0.636532
9,P102075,2026-04-20 22:49:06+00:00,twitter,julesnguyen1071,"Seriously, I got charged twice for my subscrip...",6,0,1,0.623573



 Q03 : My order has been stuck in transit with Canada Post for over a week.


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P117345,2026-08-29 19:18:24+00:00,twitter,emma.brown20,@NorthStarCoffee My order has been stuck in tr...,34,8,10,0.766164
1,P118884,2026-09-09 00:59:38+00:00,trustpilot,yaraortiz87,Update: My order has been stuck in transit for...,2,0,0,0.749343
2,P102521,2026-04-24 23:55:06+00:00,twitter,maxkhan1837,@NorthStarCoffee My order has been stuck in tr...,7,1,2,0.704261
3,P121047,2026-09-26 22:27:00+00:00,twitter,luca.gill24,Update: My order has been stuck in transit for...,5,1,2,0.649876
4,P116131,2026-08-23 08:17:07+00:00,reddit,leapatel1171,"Not happy. Still waiting for my beans, trackin...",8,2,0,0.634759
5,P101477,2026-04-14 23:28:49+00:00,reddit,max_brown,"not happy. still waiting for my beans, trackin...",12,2,0,0.604900
6,P115536,2026-08-18 17:00:52+00:00,twitter,noor_fox,Update: My order has been stuck in transit for...,7,0,2,0.603955
7,P106508,2026-06-02 10:41:28+00:00,instagram,tarafox987,My order has been stuck in transit for 6 days ...,33,10,4,0.598226
8,P100926,2026-04-09 13:30:13+00:00,instagram,maxkhan1837,My order has been stuck in transit for 6 days ...,28,4,1,0.575733
9,P102545,2026-04-25 07:33:15+00:00,trustpilot,sarasmith858,PSA: Package marked delivered but nothing at m...,0,0,0,0.567365


... (12 outputs left out)


 Q10 : Paid for my order but it vanished from my account.


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P117179,2026-08-29 02:57:05+00:00,trustpilot,nina_gill,"Seriously, my order vanished from the app afte...",1,0,0,0.369614
1,P116013,2026-08-22 01:08:06+00:00,twitter,chloe_wong,@NorthStarCoffee My order vanished from the ap...,7,1,3,0.282982
2,P120547,2026-09-21 22:38:08+00:00,twitter,ben.chen1,@NorthStarCoffee PSA: My order vanished from t...,3,1,0,0.271371
3,P116811,2026-08-26 13:30:59+00:00,twitter,hugor668,"App crashed at payment, got charged but no ord...",8,0,2,0.252421
4,P119674,2026-09-15 18:13:24+00:00,reddit,emma.singh51,"psa: app crashed at payment, got charged but n...",11,4,1,0.243693
5,P119645,2026-09-15 14:50:34+00:00,twitter,sam_patel,"App crashed at payment, got charged but no ord...",1,0,1,0.237333
6,P119224,2026-09-12 10:42:48+00:00,reddit,liam.patel62,PSA: App crashes when I open my subscription p...,9,1,0,0.233180
7,P116284,2026-08-23 23:26:26+00:00,trustpilot,emma.brown20,Ugh. My package vanished from the app after it...,2,0,0,0.232984
8,P120569,2026-09-22 07:49:58+00:00,twitter,dev.hall1,"App crashed at payment, got charged but no pac...",21,3,3,0.228950
9,P118472,2026-09-06 08:58:46+00:00,twitter,sara.chen40,"App crashed at payment, got charged but no ord...",15,2,2,0.226916


,query_id,relevant_count_out_of_10,precision_at_10
0,Q01,None,None
1,Q02,None,None
2,Q03,None,None
3,Q04,None,None
4,Q05,None,None
5,Q06,None,None
6,Q07,None,None
7,Q08,None,None
8,Q09,None,None
9,Q10,None,None


Recommendation analysis completed!
Precision@10 = relevant results divided by 10.


### 5a. Q01 before and after removing the suspicious burst

In [37]:
# Create burst_ids from your suspicious burst DataFrame
# Change suspicious_burst to the actual DataFrame name if yours is different.

if "suspicious_burst" in globals():
    burst_ids = set(
        suspicious_burst["post_id"].astype(str)
    )
else:
    burst_ids = set()

# Get Q01 query
q01 = queries.loc[
    queries["query_id"].eq("Q01"),
    query_text_col
].iloc[0]

# Recommendations before and after burst removal
q01_before = recommend(q01, df, 10)
q01_after = recommend(q01, df_clean, 10)

print("Q01 before burst removal:")
display(q01_before)

print("Q01 after burst removal:")
display(q01_after)

print(
    "Burst posts in Q01 top 10 before removal:",
    int(q01_before["post_id"].astype(str).isin(burst_ids).sum())
)

Q01 before burst removal:


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P108133,2026-06-12 13:20:24+00:00,trustpilot,roast_king8040,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
1,P108086,2026-06-12 11:49:44+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
2,P107487,2026-06-11 01:42:57+00:00,trustpilot,beanaddict8237,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
3,P107987,2026-06-12 06:31:14+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
4,P107794,2026-06-11 17:49:01+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
5,P107589,2026-06-11 07:32:30+00:00,trustpilot,latteaddict7754,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
6,P107577,2026-06-11 07:04:47+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
7,P107467,2026-06-11 00:52:52+00:00,trustpilot,java_lover8730,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
8,P111801,2026-07-14 22:50:51+00:00,reddit,sara_lee,anyone tried northstar coffee? is northstar co...,5,0,0,0.263790
9,P120306,2026-09-20 11:46:45+00:00,instagram,priya.martin42,"Coffee and stargazing, looking for the North S...",7,2,1,0.251035


Q01 after burst removal:


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P108133,2026-06-12 13:20:24+00:00,trustpilot,roast_king8040,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
1,P108086,2026-06-12 11:49:44+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
2,P107487,2026-06-11 01:42:57+00:00,trustpilot,beanaddict8237,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
3,P107987,2026-06-12 06:31:14+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
4,P107794,2026-06-11 17:49:01+00:00,trustpilot,coffee_daily3464,Best coffee I have ever tasted! NorthStar Coff...,2,0,0,0.272092
5,P107589,2026-06-11 07:32:30+00:00,trustpilot,latteaddict7754,Best coffee I have ever tasted! NorthStar Coff...,0,0,0,0.272092
6,P107577,2026-06-11 07:04:47+00:00,trustpilot,roast_king8958,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
7,P107467,2026-06-11 00:52:52+00:00,trustpilot,java_lover8730,Best coffee I have ever tasted! NorthStar Coff...,1,0,0,0.272092
8,P111801,2026-07-14 22:50:51+00:00,reddit,sara_lee,anyone tried northstar coffee? is northstar co...,5,0,0,0.263790
9,P120306,2026-09-20 11:46:45+00:00,instagram,priya.martin42,"Coffee and stargazing, looking for the North S...",7,2,1,0.251035


Burst posts in Q01 top 10 before removal: 0


### 5b. Q06 usefulness check

In [38]:
if "Q06" in all_recs:
    print("Q06 query:", queries.loc[queries.query_id.eq("Q06"), query_text_col].iloc[0])
    display(all_recs["Q06"])
    print("Possible improvement: add diversity so the top results do not all come from the same author or nearly identical wording.")

Q06 query: $25 for a small bag of beans feels like a lot. Is it worth the price?


,post_id,created_at,platform,author,text,likes,replies,shares,similarity
0,P103327,2026-05-03 14:19:10+00:00,reddit,max_yow,"Not worth the price anymore, the bags keep get...",5,1,0,0.274146
1,P109483,2026-06-23 16:24:55+00:00,trustpilot,emma.brown20,The subscription discount makes it worth it.,3,0,0,0.267197
2,P116769,2026-08-26 09:48:33+00:00,twitter,nina.wong22,"Not worth the price anymore, the bags keep get...",8,1,4,0.256890
3,P113479,2026-07-29 22:33:58+00:00,trustpilot,lea_smith,Update: The subscription discount makes it wor...,0,0,0,0.254116
4,P105873,2026-05-27 21:33:55+00:00,trustpilot,taramartin1267,"Ugh. Not worth the price anymore, the bags kee...",1,0,0,0.253725
5,P115145,2026-08-15 19:24:58+00:00,twitter,chloe_hikes,"PSA: not worth the price anymore, the bags kee...",3,1,5,0.252435
6,P107908,2026-06-11 23:51:17+00:00,reddit,sam.byrne8,$26.0 for a 340 g bag is way too expensive. no...,8,2,0,0.246098
7,P101522,2026-04-15 13:16:51+00:00,twitter,yara_byrne,"Ugh. Not worth the price anymore, the bags kee...",50,8,8,0.239861
8,P114184,2026-08-06 00:05:47+00:00,twitter,amirchen1092,@NorthStarCoffee Not happy. Not worth the pric...,5,0,0,0.236050
9,P110871,2026-07-07 03:01:27+00:00,trustpilot,ben.k85,The subscription discount makes it worth it. T...,0,0,0,0.235697


Possible improvement: add diversity so the top results do not all come from the same author or nearly identical wording.


## 6. Recommendations and reply priority (Task 5)

The reply queue looks for customers who say they may leave, cancel, switch, or stop ordering. It ranks posts by total engagement. The last 14 days means the last 14 days present in this dataset, not today's date.

In [39]:
df_clean["engagement"] = df_clean[["likes","replies","shares"]].fillna(0).sum(axis=1)
max_date = df_clean.created_at.max()
last_14_start = max_date - pd.Timedelta(days=14)
leaving_pattern = r"cancel|switch(ing)?|leave|leaving|not order(ing)? again|won't order|will not order|stop buying|never buy again|done with"
reply_queue = df_clean[
    (df_clean.created_at >= last_14_start) &
    df_clean.text.str.contains(leaving_pattern, case=False, regex=True, na=False)
].copy()
reply_queue = reply_queue.sort_values("engagement", ascending=False)
display(reply_queue[["post_id","created_at","platform","author","text","likes","replies","shares","engagement"]].head(10))
print("Posts matching leaving-intent rules:", len(reply_queue))

C:\Users\mahir\AppData\Local\Temp\ipykernel_31420\371677930.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_clean.text.str.contains(leaving_pattern, case=False, regex=True, na=False)


,post_id,created_at,platform,author,text,likes,replies,shares,engagement
3064,P121106,2026-09-27 13:50:33+00:00,reddit,omar_r,"cannot log in to the app since the update, kee...",92,18,10,120
3019,P120800,2026-09-24 19:45:15+00:00,twitter,nina.wong22,Update: My subscription renewed without any re...,44,8,15,67
3122,P121508,2026-09-30 11:43:56+00:00,instagram,leahall1500,"Okay, really impressed with the flavour of the...",47,2,3,52
2922,P120119,2026-09-18 22:14:50+00:00,twitter,leo_k,"@NorthStarCoffee Seriously, the app crashed du...",23,2,5,30
2887,P119873,2026-09-16 23:47:10+00:00,instagram,noor_wong,Ugh. I got charged twice for my subscription t...,25,3,1,29
2991,P120605,2026-09-22 22:17:26+00:00,reddit,liam_martin,Amazing how easy it is to subscribe and how im...,22,2,3,27
3039,P120933,2026-09-26 08:11:02+00:00,reddit,finn_reads,app crashes when i open my subscription page s...,19,5,2,26
3043,P120959,2026-09-26 10:58:02+00:00,instagram,chloekhan1564,"Seriously, sorry but the decaf is sour and fla...",19,3,3,25
2956,P120356,2026-09-20 18:27:40+00:00,twitter,amir.brown36,@NorthStarCoffee Ugh. With shipping it came to...,13,3,3,19
2955,P120354,2026-09-20 17:36:30+00:00,instagram,dev.wong3,Update: Cancelling the subscription is impossi...,13,4,0,17


Posts matching leaving-intent rules: 32


## 7. Monitoring and ethics

In [40]:
def theme_spike_check(data, theme_id, baseline_weeks=4, increase_threshold=0.50):
    weekly_counts = data.groupby(["week","theme_id"]).size().unstack(fill_value=0)
    if theme_id not in weekly_counts.columns or len(weekly_counts) <= baseline_weeks:
        return None
    weekly_share = weekly_counts.div(weekly_counts.sum(axis=1), axis=0)[theme_id]
    alerts = []
    for i in range(baseline_weeks, len(weekly_share)):
        baseline = weekly_share.iloc[i-baseline_weeks:i].mean()
        current = weekly_share.iloc[i]
        if baseline > 0 and (current - baseline) / baseline >= increase_threshold:
            alerts.append({"week": weekly_share.index[i], "baseline_share": baseline,
                           "current_share": current, "relative_increase": (current-baseline)/baseline})
    return pd.DataFrame(alerts)

print("Example monitoring rule: alert the social care lead when a theme's weekly share is at least 50% above its average over the previous four weeks.")
print("The alert should include the theme name, week, current share, baseline share, and sample posts.")

Example monitoring rule: alert the social care lead when a theme's weekly share is at least 50% above its average over the previous four weeks.
The alert should include the theme name, week, current share, baseline share, and sample posts.


**Ethics and privacy (report text):** Social posts can contain usernames and details that identify real people. The team should only collect information needed for customer support, limit access, and remove or mask usernames in reports. In Canada, the organization should consider its obligations under PIPEDA and applicable provincial privacy rules. If an external language model is used, customer text should not be sent unless the organization has approved the service and the data-sharing terms.

**Limitations:** A keyword scrape may miss customers who use different words or discuss the product without naming it. Sentiment tools can misunderstand sarcasm, slang, mixed feelings, and short posts. Similar reviews are a warning sign, not proof of fake activity. The recommender finds text similarity, not necessarily the best answer or the most helpful post.

## 8. Optional bonus — mention network

This section is optional. It extracts account mentions and builds a directed graph from author to mentioned account.

In [41]:
try:
    import networkx as nx
    G = nx.DiGraph()
    for _, row in df.iterrows():
        author = str(row["author"]).strip()
        mentions = re.findall(r"@([A-Za-z0-9_]+)", str(row["text"]))
        for mentioned in mentions:
            G.add_edge(author, mentioned)
    print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
    indeg = dict(G.in_degree())
    between = nx.betweenness_centrality(G, normalized=True)
    brand = "NorthStarCoffee"
    accounts = [n for n in G.nodes if n.casefold() != brand.casefold()]
    print("Top in-degree:", sorted([(n, indeg.get(n,0)) for n in accounts], key=lambda x:x[1], reverse=True)[:10])
    print("Top betweenness:", sorted([(n, between.get(n,0)) for n in accounts], key=lambda x:x[1], reverse=True)[:10])
    G.remove_node(brand) if brand in G else None
    print("Largest weakly connected components:",
          sorted([len(c) for c in nx.weakly_connected_components(G)], reverse=True)[:10])
except ImportError:
    print("Install networkx to run the optional network section.")

Nodes: 556 Edges: 690
Top in-degree: [('thegrindreview', 27), ('maplebrewmom', 20), ('beanscene_ottawa', 18), ('java_jules_ca', 15), ('emma', 8), ('coffee_daily4533', 8), ('coffee_daily3464', 4), ('java_king5012', 4), ('javaaddict6574', 4), ('javafan1691', 4)]
Top betweenness: [('coffee_daily4533', 0.0006949187454602617), ('roast_king9104', 0.0005360956624494531), ('java_king5012', 0.000504656280829566), ('beanaddict8237', 0.0004338634663544411), ('coffeefan4899', 0.00037906137184115523), ('roast_king8040', 0.00035694539304647607), ('coffee_daily3464', 0.0003458874036491365), ('bean_daily7643', 0.00031650784358365583), ('brew_king2380', 0.0003141227870469748), ('javafan1691', 0.0003100573497685411)]
Largest weakly connected components: [119, 30, 4, 3, 3, 3, 3, 3, 3, 2]


## 9. Key Findings Table

Run the notebook, review the flagged posts, rename the themes, and set the emerging theme ID. Do not submit placeholder values. The assignment data files were not included with the starter notebook in this chat, so these values are calculated only when you run it with the supplied CSV files.

In [43]:
import pandas as pd
import numpy as np

# Make sure dates are in datetime format
df_clean["created_at"] = pd.to_datetime(
    df_clean["created_at"], errors="coerce"
)

# Find the rating column
rating_col = next(
    (c for c in ["trustpilot_rating", "rating", "review_rating", "stars"]
     if c in df_clean.columns),
    None
)

if rating_col is None:
    raise ValueError(
        "Rating column not found. Available columns: "
        + str(df_clean.columns.tolist())
    )

# Select June posts
june_data = df_clean[
    df_clean["created_at"].dt.month.eq(6)
].copy()

# Keep only valid numeric ratings
june_data[rating_col] = pd.to_numeric(
    june_data[rating_col], errors="coerce"
)
june_data = june_data.dropna(subset=[rating_col])

# Average before removing burst candidates
june_avg_with = june_data[rating_col].mean()

# Average after removing burst candidates
june_without_burst = june_data[
    ~june_data["post_id"].astype(str).isin(burst_ids)
]

june_avg_without_candidates = june_without_burst[rating_col].mean()

print("June Trustpilot average before burst removal:",
      round(june_avg_with, 3))

print("June Trustpilot average after burst removal:",
      round(june_avg_without_candidates, 3))

print("June reviews:", len(june_data))
print("June reviews after burst removal:", len(june_without_burst))

June Trustpilot average before burst removal: 4.076
June Trustpilot average after burst removal: 4.076
June reviews: 157
June reviews after burst removal: 157
